In [1]:
import os, sys
# Ensure project root is in sys.path for src.* imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [2]:
from src.experiment_notebook import setup_experiment, fine_tune_and_evaluate
import torch

## Prova amb fairness_loss del TPR

In [3]:
def fairness_loss(labels, genders, probs, alpha=0.0, **kwargs):
    """
    Calculate fairness penalty (difference in TPR between genders).
    Returns a tensor (in the same device as `probs`) representing the absolute difference in TPR.
    If alpha <= 0.0 or there are not enough positive samples, returns 0.0.
    """
    if alpha <= 0.0:
        return torch.tensor(0.0, device=probs.device)
    pos_mask = labels == 1
    if pos_mask.sum() > 0:
        genders_pos = genders[pos_mask]
        probs_pos = probs[pos_mask]
        g1_mask = genders_pos == 0
        g2_mask = genders_pos == 1
        g1_mean = probs_pos[g1_mask].mean() if g1_mask.sum() > 0 else torch.tensor(0.0).to(probs.device)
        g2_mean = probs_pos[g2_mask].mean() if g2_mask.sum() > 0 else torch.tensor(0.0).to(probs.device)
        return (g1_mean - g2_mean).pow(2)
    else:
        return torch.tensor(0.0, device=probs.device)

In [4]:
ctx = setup_experiment(
    prev2last_layer_size=256,
    batch_size=32,
    lr=1e-5,
    model_type='patient',
    weight_decay=1e-4,
    early_stopping_patience=5,
    fairness_loss_fn=fairness_loss,
    run_name='tpr_parity'
)

Device: cpu
Config: alpha=0.0, layer_size=256, batch_size=32, lr=1e-05
Model: PatientFeatureExtractor (dropout=0.3)
Weight decay: 0.0001
Data: 1829 train / 458 test samples
Results dir: ./results_NIH/tpr_parity_20260325_123613


In [5]:
print("Training model...\n")
results = fine_tune_and_evaluate(ctx, epochs=50, alpha=0.5)

Training model...

  Loss: 0.6157 + 0.5*fairness penalty: 0.0115 # Loss gradient norms: 4.1373 # CE-Loss grad. norms: 4.1774
  Epoch 1, Train Loss: 0.6215, Val Loss: 0.0389
  Loss: 0.4807 + 0.5*fairness penalty: 0.0265 # Loss gradient norms: 4.3997 # CE-Loss grad. norms: 4.2425
  Epoch 2, Train Loss: 0.4939, Val Loss: 0.0365
  Loss: 0.4129 + 0.5*fairness penalty: 0.0170 # Loss gradient norms: 4.9174 # CE-Loss grad. norms: 4.6584
  Epoch 3, Train Loss: 0.4214, Val Loss: 0.0337
  Loss: 0.3663 + 0.5*fairness penalty: 0.0193 # Loss gradient norms: 5.1347 # CE-Loss grad. norms: 4.7808
  Epoch 4, Train Loss: 0.3759, Val Loss: 0.0342
  Loss: 0.3069 + 0.5*fairness penalty: 0.0155 # Loss gradient norms: 5.2889 # CE-Loss grad. norms: 4.9958
  Epoch 5, Train Loss: 0.3147, Val Loss: 0.0359
  Loss: 0.2753 + 0.5*fairness penalty: 0.0170 # Loss gradient norms: 5.6448 # CE-Loss grad. norms: 5.2648
  Epoch 6, Train Loss: 0.2838, Val Loss: 0.0442
  Loss: 0.2220 + 0.5*fairness penalty: 0.0192 # Loss grad

## Prova amb Demographic Parity (panalització dels positius, independentment del ground truth)

In [6]:
def demographic_parity_loss(labels, genders, probs, alpha=0.0, **kwargs):
    """
    Penalizes the difference in positive prediction rates between gender groups,
    regardless of the true label. This enforces that P(Y_hat=1 | G=0) ≈ P(Y_hat=1 | G=1).
    """
    if alpha <= 0.0:
        return torch.tensor(0.0, device=probs.device)

    g1_mask = genders == 0
    g2_mask = genders == 1

    g1_count = g1_mask.sum()
    g2_count = g2_mask.sum()

    if g1_count == 0 or g2_count == 0:
        return torch.tensor(0.0, device=probs.device)

    g1_pred_rate = probs[g1_mask].mean()
    g2_pred_rate = probs[g2_mask].mean()

    return (g1_pred_rate - g2_pred_rate).pow(2)

In [7]:
ctx = setup_experiment(
    prev2last_layer_size=256,
    batch_size=32,
    lr=1e-5,
    model_type='patient',
    weight_decay=1e-4,
    early_stopping_patience=5,
    fairness_loss_fn=demographic_parity_loss,
    run_name='demographic_parity'
)

Device: cpu
Config: alpha=0.0, layer_size=256, batch_size=32, lr=1e-05
Model: PatientFeatureExtractor (dropout=0.3)
Weight decay: 0.0001
Data: 1829 train / 458 test samples
Results dir: ./results_NIH/demographic_parity_20260325_131040


In [8]:
print("Training model...\n")
results = fine_tune_and_evaluate(ctx, epochs=50, alpha=0.5)

Training model...

  Loss: 0.6154 + 0.5*fairness penalty: 0.0018 # Loss gradient norms: 4.1743 # CE-Loss grad. norms: 4.1888
  Epoch 1, Train Loss: 0.6163, Val Loss: 0.0323
  Loss: 0.4778 + 0.5*fairness penalty: 0.0075 # Loss gradient norms: 4.2398 # CE-Loss grad. norms: 4.2379
  Epoch 2, Train Loss: 0.4815, Val Loss: 0.0297
  Loss: 0.4117 + 0.5*fairness penalty: 0.0141 # Loss gradient norms: 4.6041 # CE-Loss grad. norms: 4.6126
  Epoch 3, Train Loss: 0.4188, Val Loss: 0.0266
  Loss: 0.3657 + 0.5*fairness penalty: 0.0165 # Loss gradient norms: 4.9238 # CE-Loss grad. norms: 4.8535
  Epoch 4, Train Loss: 0.3740, Val Loss: 0.0273
  Loss: 0.3077 + 0.5*fairness penalty: 0.0223 # Loss gradient norms: 5.1397 # CE-Loss grad. norms: 5.0585
  Epoch 5, Train Loss: 0.3189, Val Loss: 0.0270
  Loss: 0.2754 + 0.5*fairness penalty: 0.0245 # Loss gradient norms: 5.2618 # CE-Loss grad. norms: 5.2455
  Epoch 6, Train Loss: 0.2877, Val Loss: 0.0322
  Loss: 0.2237 + 0.5*fairness penalty: 0.0255 # Loss grad

## Prova amb Disentanglement (HSIC)
Directament penalitza la correlació entre les representacions internes de la xarxa amb el gènere amb un mètode estadístic anomenat Hilbert-Schmidt Independence Criterion.

In [9]:
def rbf_kernel_matrix(x, sigma=None):
    """
    Computes the RBF (Gaussian) kernel matrix K where K_ij = exp(-||x_i - x_j||^2 / (2 * sigma^2)).
    If sigma is None, uses the median heuristic: sigma = median of pairwise distances.
    x: tensor of shape (n, d)
    Returns: kernel matrix of shape (n, n)
    """
    dists_sq = torch.cdist(x, x, p=2).pow(2)

    if sigma is None:
        mask = dists_sq > 0
        if mask.sum() > 0:
            sigma = dists_sq[mask].median().sqrt().clamp(min=1e-5)
        else:
            sigma = torch.tensor(1.0, device=x.device)

    return torch.exp(-dists_sq / (2 * sigma ** 2))


def hsic(K, L):
    """
    Computes the empirical HSIC (Hilbert-Schmidt Independence Criterion) between
    two kernel matrices K and L of shape (n, n).
    HSIC ≈ 0 means the two variables are independent in the RKHS.
    """
    n = K.shape[0]
    if n < 4:
        return torch.tensor(0.0, device=K.device)

    K_row_mean = K.mean(dim=1, keepdim=True)
    K_col_mean = K.mean(dim=0, keepdim=True)
    K_mean = K.mean()
    K_centered = K - K_row_mean - K_col_mean + K_mean

    L_row_mean = L.mean(dim=1, keepdim=True)
    L_col_mean = L.mean(dim=0, keepdim=True)
    L_mean = L.mean()
    L_centered = L - L_row_mean - L_col_mean + L_mean

    return (K_centered * L_centered).sum() / ((n - 1) ** 2)


def _hsic_one_layer(features_2d, gender_vec, L_gender):
    """Compute HSIC between one layer's features and the precomputed gender kernel."""
    K = rbf_kernel_matrix(features_2d)
    return hsic(K, L_gender)


def disentanglement_loss(labels, genders, probs, alpha=0.0, features=None, **kwargs):
    """
    Penalizes statistical dependence between intermediate feature representations
    and the sensitive attribute (gender) using HSIC with RBF kernels.

    Compatible with the unified fairness_loss_fn interface:
        fairness_loss_fn(labels=..., genders=..., probs=..., alpha=..., features=...)

    Parameters
    ----------
    labels : ignored (present for interface compatibility)
    genders : tensor of shape (batch_size,) — binary sensitive attribute
    probs : ignored (present for interface compatibility)
    alpha : float — penalty weight (if 0, returns 0)
    features : list of tensors — intermediate activations from model.features_all.
               Each tensor has shape (batch_size, ...).

    Returns
    -------
    Scalar tensor: mean HSIC across all layers.
    """
    if alpha <= 0.0:
        return torch.tensor(0.0, device=probs.device)

    if features is None or len(features) == 0:
        return torch.tensor(0.0, device=probs.device)

    n = genders.shape[0]
    if n < 4:
        return torch.tensor(0.0, device=probs.device)

    # Precompute the gender kernel once (shared across all layers)
    gender_vec = genders.float().unsqueeze(1)  # (n, 1)
    L_gender = rbf_kernel_matrix(gender_vec)

    # Accumulate HSIC for each layer
    total_hsic = torch.tensor(0.0, device=probs.device)
    num_layers = 0

    for feat in features:
        if feat is None or feat.shape[0] != n:
            continue
        # Flatten conv features: (batch, C, H, W) -> (batch, C*H*W)
        feat_2d = feat.view(n, -1) if feat.dim() > 2 else feat
        total_hsic = total_hsic + _hsic_one_layer(feat_2d, gender_vec, L_gender)
        num_layers += 1

    if num_layers == 0:
        return torch.tensor(0.0, device=probs.device)

    return total_hsic / num_layers

In [10]:
ctx = setup_experiment(
    prev2last_layer_size=256,
    batch_size=32,
    lr=1e-5,
    model_type='patient',
    weight_decay=1e-4,
    early_stopping_patience=5,
    fairness_loss_fn=disentanglement_loss,
    run_name='hsic_disentanglement'
)

Device: cpu
Config: alpha=0.0, layer_size=256, batch_size=32, lr=1e-05
Model: PatientFeatureExtractor (dropout=0.3)
Weight decay: 0.0001
Data: 1829 train / 458 test samples
Results dir: ./results_NIH/hsic_disentanglement_20260325_133925


In [11]:
print("Training model...\n")
results = fine_tune_and_evaluate(ctx, epochs=50, alpha=0.5)

Training model...

  Loss: 0.6150 + 0.5*fairness penalty: 0.0032 # Loss gradient norms: 4.2072 # CE-Loss grad. norms: 4.2068
  Epoch 1, Train Loss: 0.6166, Val Loss: 0.0323
  Loss: 0.4762 + 0.5*fairness penalty: 0.0031 # Loss gradient norms: 4.2647 # CE-Loss grad. norms: 4.2644
  Epoch 2, Train Loss: 0.4777, Val Loss: 0.0284
  Loss: 0.4096 + 0.5*fairness penalty: 0.0033 # Loss gradient norms: 4.6423 # CE-Loss grad. norms: 4.6427
  Epoch 3, Train Loss: 0.4113, Val Loss: 0.0247
  Loss: 0.3630 + 0.5*fairness penalty: 0.0031 # Loss gradient norms: 5.0239 # CE-Loss grad. norms: 5.0219
  Epoch 4, Train Loss: 0.3645, Val Loss: 0.0231
  Loss: 0.3035 + 0.5*fairness penalty: 0.0033 # Loss gradient norms: 5.0749 # CE-Loss grad. norms: 5.0746
  Epoch 5, Train Loss: 0.3051, Val Loss: 0.0211
  Loss: 0.2709 + 0.5*fairness penalty: 0.0033 # Loss gradient norms: 5.3280 # CE-Loss grad. norms: 5.3292
  Epoch 6, Train Loss: 0.2726, Val Loss: 0.0237
  Loss: 0.2168 + 0.5*fairness penalty: 0.0031 # Loss grad